
### Install das Libs


In [0]:
%pip install pandas matplotlib seaborn numpy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np


### Carregamento das Bases

In [0]:
# Caminho base
path = "/Workspace/Users/kaetanokako23@gmail.com/pos tech/data/"

# Leitura dos arquivos
customers = pd.read_csv(path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(path + "olist_orders_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")


### Tratamentos Iniciais

In [0]:
# Converter datas
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"], errors="coerce")
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"], errors="coerce")


In [0]:
# Conferencia de tamanhos
dfs = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")


customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


### tratamentos de estrutura

In [0]:
products = products.merge(
    category_translation,
    how="left",
    on="product_category_name"
)

products["product_category"] = products["product_category_name_english"].fillna(products["product_category_name"])


In [0]:
# Join Pagamentos

payments_agg = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_types_n=("payment_type", "nunique")
    )
)

payment_main = (
    order_payments
    .groupby(["order_id", "payment_type"], as_index=False)["payment_value"]
    .sum()
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "payment_type_main"})
)

payments_agg = payments_agg.merge(payment_main, on="order_id", how="left")


In [0]:
# join Reviwes
reviews_agg = (
    order_reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max"),
        review_count=("review_id", "nunique")
    )
)


In [0]:
# join localizacao

geo_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean")
    )
)


In [0]:
df = order_items.merge(orders, how="left", on="order_id")
df = df.merge(customers, how="left", on="customer_id")
df = df.merge(products, how="left", on="product_id")
df = df.merge(sellers, how="left", on="seller_id")
df = df.merge(payments_agg, how="left", on="order_id")
df = df.merge(reviews_agg, how="left", on="order_id")


In [0]:
# Add as customer location 
df = df.merge(
    geo_agg,
    how="left",
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix"
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
}).drop(columns=["geolocation_zip_code_prefix"], errors="ignore")

geo_seller = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})

df = df.merge(geo_seller, how="left", on="seller_zip_code_prefix")


In [0]:
# Criacao das Metricas

df["item_revenue"] = df["price"]
df["item_freight"] = df["freight_value"]
df["item_total"] = df["price"] + df["freight_value"]

df["approval_time_hours"] = (
    (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600
)

df["carrier_time_days"] = (
    (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.total_seconds() / 86400
)

df["delivery_time_days"] = (
    (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
)

df["estimated_time_days"] = (
    (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
)

df["delay_days"] = (
    (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.total_seconds() / 86400
)

df["is_delayed"] = np.where(df["delay_days"] > 0, 1, 0)
df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["purchase_quarter"] = df["order_purchase_timestamp"].dt.to_period("Q").astype(str)


In [0]:
# Um filtro para forcarmos nos pedidos entregues 
df_delivered = df[df["order_status"] == "delivered"].copy()

In [0]:
df_delivered.to_csv("/Workspace/Users/kaetanokako23@gmail.com/pos tech/data/base_tratada.csv", index=False)


In [0]:
spark.createDataFrame(df_delivered).write.mode("overwrite").saveAsTable("olist_base_analitica")